In [1]:
from specml.data import SpectroscopicData
from specml.visualization import plotting
from specml.processing.masking import create_mask_from_area, apply_mask, rebuild_tissue_mask
from specml.processing.preprocessing import vector_norm, adjust_spectra_min_to_zero
import numpy as np

from specml.data.multi_spectroscopic_data import MultiFileSpectroscopicData
# from specml.feature_engineering.dim_reduction import PCAAnalysis
from specml.ml_models.clustering import KMeansClustering, ClusteringResult

In [2]:
file_paths = [
    'E:/Lung_Data_8cm/6F-6504-01/6F-6504-01_top2/6f-6504-01_top2.dmt',
    'E:/Lung_Data_8cm/2F-2501-01/1F-2501-01_top/1f-2501-01_top.dmt',
    'E:/Lung_Data_8cm/1F-1504-01/1F-1504-01_top/1f-1504-01_top.dmt',
]

multi_spec_data = MultiFileSpectroscopicData(file_paths)

# Preprocess all data
multi_spec_data.easy_preprocess(area_threshold=4.0)
combined_data = multi_spec_data.get_combined_data()

INFO:specml.data.loaders:Data loaded from E:/Lung_Data_8cm/6F-6504-01/6F-6504-01_top2/6f-6504-01_top2.dmt with shape (512, 1664, 754)
INFO:specml.data.spectroscopic_data:Data flattened and mask applied. Shape of processed data: (851968, 754)
INFO:specml.data.spectroscopic_data:{'filename': 'E:/Lung_Data_8cm/6F-6504-01/6F-6504-01_top2/6f-6504-01_top2.dmt', 'xpixels': 1664, 'ypixels': 512, 'xtiles': 13, 'ytiles': 4, 'numpts': 754, 'fpasize': 128, 'original_shape': (512, 1664, 754)}
INFO:specml.data.loaders:Data loaded from E:/Lung_Data_8cm/2F-2501-01/1F-2501-01_top/1f-2501-01_top.dmt with shape (512, 1792, 754)
INFO:specml.data.spectroscopic_data:Data flattened and mask applied. Shape of processed data: (917504, 754)
INFO:specml.data.spectroscopic_data:{'filename': 'E:/Lung_Data_8cm/2F-2501-01/1F-2501-01_top/1f-2501-01_top.dmt', 'xpixels': 1792, 'ypixels': 512, 'xtiles': 14, 'ytiles': 4, 'numpts': 754, 'fpasize': 128, 'original_shape': (512, 1792, 754)}
INFO:specml.data.loaders:Data load

In [10]:
from sklearn.decomposition import PCA
from specml.feature_engineering.dim_reduction import apply_pca

pca = PCA(n_components=5)
pca_result = pca.fit_transform(combined_data)

kmeans = KMeansClustering(n_clusters=5, scale_data=True)

clustering_algorithm = kmeans
# Fit the clustering algorithm
cluster_labels = clustering_algorithm.fit_predict(combined_data)

# Create ClusteringResult
clustering_result = ClusteringResult(
    cluster_labels=cluster_labels,
    model=clustering_algorithm.get_model(),
    mask=multi_spec_data.get_combined_mask(),
    original_shape=[(512, 1664), (512, 1792), (512, 1536)]
    )

In [11]:
# Reconstruct cluster labels into images
cluster_images = clustering_result.to_images()

ValueError: NumPy boolean array indexing assignment cannot assign 810492 input values to the 319784 output values where the mask is true

In [ ]:
from specml.ml_models.clustering import display_images
# Plot cluster images
titles = [f"Clustering Result for File {idx+1}" for idx in range(len(cluster_images))]
display_images(cluster_images, [512,512,512], [1664, 1792,1536], titles=titles, color_bar=True)